# Sales Data Analysis

## 1. Project Introduction
This project analyzes a synthetic sales dataset to extract meaningful business insights. The purpose is to demonstrate data cleaning, exploratory data analysis (EDA), visualization, and SQL skills using Python, Pandas, Matplotlib, Seaborn, and SQLite.

## 2. Business Questions
* What is the total revenue and total number of orders?
* What is the average order value?
* Which products generate the most revenue?
* How are sales distributed across different regions and categories?
* What is the monthly sales trend?
* Who are the top customers by revenue?

## 3. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

# Display plots in notebook
%matplotlib inline

## 4. Loading the Raw Dataset

In [ ]:
raw_df = pd.read_csv('../data/raw/sales_data.csv')
raw_df.head()

## 5. Understanding the Dataset

In [ ]:
print(f'Shape: {raw_df.shape}')
raw_df.info()

## 6. Data-Quality Checks

In [ ]:
# Check missing values
print('Missing Values:')
print(raw_df.isnull().sum())

# Check duplicates
print(f'\nDuplicates: {raw_df.duplicated().sum()}')

## 7. Data Cleaning
We will handle missing values, correct data types, standard text, and calculate the actual sales amount.

In [ ]:
df = raw_df.copy()

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# Clean numeric columns
df['quantity'] = pd.to_numeric(df['quantity'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Standardize text
for col in ['product', 'category', 'region']:
    df[col] = df[col].astype(str).str.strip().replace('nan', np.nan).str.title()

# Handle missing values
df['product'] = df['product'].fillna('Unknown')
df['region'] = df['region'].fillna('Unknown')
df['quantity'] = df['quantity'].fillna(1.0)

# Remove duplicates
df = df.drop_duplicates()

# Recalculate sales amount
df['sales_amount'] = df['quantity'] * df['unit_price']

print(f'Cleaned Shape: {df.shape}')
df.head()

## 8. Exploratory Data Analysis & 9. Business Metrics

In [ ]:
total_revenue = df['sales_amount'].sum()
total_orders = df['order_id'].nunique()
total_quantity = df['quantity'].sum()
aov = total_revenue / total_orders

print(f'Total Revenue: ${total_revenue:,.2f}')
print(f'Total Orders: {total_orders}')
print(f'Total Quantity Sold: {total_quantity}')
print(f'Average Order Value: ${aov:,.2f}')

## 10. Data Visualizations

In [ ]:
# Monthly Sales Trend
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly_sales = df.groupby('month')['sales_amount'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_sales, x='month', y='sales_amount', marker='o')
plt.title('Monthly Sales Trend')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Revenue by Product
product_sales = df.groupby('product')['sales_amount'].sum().reset_index().sort_values('sales_amount', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=product_sales, x='sales_amount', y='product', hue='product', legend=False)
plt.title('Revenue by Product')
plt.show()

## 11. SQL Analysis using SQLite

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, if_exists='replace', index=False)

query = """
SELECT category, SUM(sales_amount) as total_revenue
FROM sales
GROUP BY category
ORDER BY total_revenue DESC;
"""

pd.read_sql_query(query, conn)

## 12. Key Business Insights
- The top performing product categories and products drive a significant portion of revenue.
- Certain regions outperform others, indicating potential for targeted marketing.
- The monthly trend highlights potential seasonality or growth periods.

## 13. Business Recommendations
- **Focus on Top Products:** Allocate more inventory and marketing budget to the highest performing products.
- **Regional Strategy:** Investigate why certain regions are underperforming and adapt strategies accordingly.
- **Customer Retention:** Target the top 5 customers with loyalty programs to maintain their high order volumes.

## 14. Conclusion
This project successfully demonstrated the end-to-end process of data cleaning, exploratory analysis, visualization, and SQL querying on a sales dataset. The insights generated can help drive data-informed business decisions.